# VisClick — Phase 4.4 / D-02 (step 1 of 2): self-supervised pretrain on the broad UI corpus

**Goal.** Adapt the source-trained YOLOv8s *backbone* to the UI image distribution via self-supervised pretraining on the ~9,646-image Zenodo unified bundle (mobile UI screens from RICO + CLAY + VINS). The output is a UI-domain-adapted backbone that the next notebook (`12_ssp_finetune.ipynb`) plugs into a YOLOv8s detector and few-shot fine-tunes on desktop.

**Corpus choice.** The proposal originally specified ~2000 unlabelled *desktop* screenshots. Live capture of that volume requires multi-day passive accumulation on the author's Windows box, which falls outside the time budget. The practical replacement is the Zenodo unified bundle the project already holds (~9.6k UI screens, mobile-rich). For SSP this is acceptable because:

- SimSiam learns *visual structure*, not domain semantics: rectangles, text blobs, icon-shaped regions, grid layouts. These are shared between mobile and desktop UI.
- 9.6k images at batch 64 gives ~150 steps per epoch, which is in the right neighbourhood for SimSiam fine-tuning of an already-pretrained backbone (we start from `best_source_v8s.pt`, not random init).
- The Limitations section names this swap explicitly: SSP pretraining is on mobile-rich UI, evaluation is on desktop, the gap is honest.

**Method.** **SimSiam** (Chen and He, 2021) on top of the source-trained CSPDarknet backbone. Chosen because:

- No negative samples, no memory bank, no large batch sizes → fits Colab Free T4.
- No momentum encoder → halves the GPU memory.
- Proven on small (≤ 10k) corpora at small batch (~32-64).

**Pipeline:**
1. Mount Drive → `git pull` → install deps.
2. Build a TwoView loader over the unlabelled Zenodo bundle at `<DRIVE>/data/unified/<split>/images/` (same path 04_assemble_source.ipynb uses). Labels are *ignored* — we treat the bundle as an unlabelled corpus for SSP.
3. Extract the YOLOv8s backbone from `best_source_v8s.pt`, attach a 3-layer projection head and a 2-layer predictor.
4. Train SimSiam for 20 epochs at batch 64, imgsz 224, two augmentation streams.
5. Save the adapted backbone to `<DRIVE>/weights/ssp/backbone_simsiam.pt` and a training-loss CSV.
6. Publish loss-log CSV to git.

**Compute reality.** 20 epochs × ~150 steps × ~0.5 s/step ≈ 25 min on T4. One Colab Free session.

**Report.** Every step prints `REPORT ...` lines.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(
    ["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics", "torch", "torchvision", "pillow", "opencv-python", "matplotlib", "pi-heif"],
    check=False,
)
import torch, torchvision, ultralytics
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("torchvision:", torchvision.__version__, "| ultralytics:", ultralytics.__version__)


## 11.1 — Build unlabelled UI corpus loader (Zenodo unified bundle)

The SSP corpus is the labelled Zenodo unified bundle (RICO + CLAY + VINS), with labels *ignored*. This sits at `<DRIVE>/data/unified/<split>/images/` — the same path `04_assemble_source.ipynb` references as `UNIFIED = <DRIVE>/data/unified`.

Each `__getitem__` emits two random augmentations of the same image (SimSiam two-view protocol). The 50-image desktop seed at `samples/desktop_seed/` is appended too, so the corpus includes a small in-distribution slice as well.


In [ ]:
import os, tarfile
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

DRIVE        = "/content/drive/MyDrive/visclick"
UNIFIED      = os.path.join(DRIVE, "data", "unified")
BUNDLES      = os.path.join(DRIVE, "data", "source_train_bundles")
SEED_DIR     = "/content/visclick/samples/desktop_seed"
IMG_EXT      = (".png", ".jpg", ".jpeg")

# Drive FUSE struggles to listdir() huge directories (>10k files), so we don't.
# Instead we extract the tiny manifests from the source_train_bundles that
# 04_assemble_source.ipynb already wrote, and resolve each filename to its
# absolute path under <DRIVE>/data/unified/<split>/images/. Opening individual
# files by full path is reliable; only the enumeration is broken.

MANIFEST_CACHE = "/content/_ssp_manifests"
os.makedirs(MANIFEST_CACHE, exist_ok=True)


def _manifest_for(split):
    out = os.path.join(MANIFEST_CACHE, f"{split}.txt")
    if os.path.isfile(out) and os.path.getsize(out) > 0:
        return out
    bundle = os.path.join(BUNDLES, f"{split}.tar.gz")
    if not os.path.isfile(bundle):
        return None
    with tarfile.open(bundle, "r:gz") as tf:
        member = None
        for m in tf.getmembers():
            if m.name.endswith(f"manifests/{split}.txt") or m.name == f"manifests/{split}.txt":
                member = m; break
        if member is None:
            return None
        f = tf.extractfile(member)
        with open(out, "wb") as fh:
            fh.write(f.read())
    return out


def collect_corpus():
    paths = []
    for split in ("train", "val", "test"):
        mf = _manifest_for(split)
        if mf is None:
            print(f"  {split}: no manifest (skipped)")
            continue
        with open(mf) as fh:
            names = [ln.strip() for ln in fh if ln.strip()]
        img_dir = os.path.join(UNIFIED, split, "images")
        before = len(paths)
        for fn in names:
            paths.append(os.path.join(img_dir, fn))
        print(f"  {split}: {len(paths) - before} paths from manifest")
    if os.path.isdir(SEED_DIR):
        for f in os.listdir(SEED_DIR):
            if f.lower().endswith(IMG_EXT):
                paths.append(os.path.join(SEED_DIR, f))
    return sorted(set(paths))


CORPUS = collect_corpus()
print(f"REPORT corpus | size = {len(CORPUS)} | head = {CORPUS[:2]}")
assert len(CORPUS) >= 1000, (
    f"Corpus too small ({len(CORPUS)}). Expected manifests under <DRIVE>/data/source_train_bundles/. "
    f"Re-run 04_assemble_source.ipynb if those bundles are missing."
)

IMG_SIZE = 224  # standard SimSiam input; smaller than 256 to save memory at batch 64.


def simsiam_aug():
    return T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.RandomApply([T.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))], p=0.5),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


class TwoViewDataset(Dataset):
    def __init__(self, paths, aug):
        self.paths = paths
        self.aug = aug

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        try:
            with Image.open(p) as im:
                im = im.convert("RGB")
        except Exception:
            # Drive FUSE can throw transient errors; return a noise pair so the
            # batch survives. The augmentation pipeline handles arbitrary RGB.
            im = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        return self.aug(im), self.aug(im)


BATCH = 64
loader = DataLoader(
    TwoViewDataset(CORPUS, simsiam_aug()),
    batch_size=BATCH, shuffle=True, num_workers=2,
    pin_memory=True, drop_last=True, persistent_workers=True,
)
print(f"REPORT loader | batch = {BATCH} | steps_per_epoch = {len(loader)}")


## 11.2 — Extract the YOLOv8s backbone from `best_source_v8s.pt`

The Ultralytics YOLO model wraps a `DetectionModel` whose first 9 modules are the CSPDarknet backbone (P1-P5 + SPPF). We yank that nn.Sequential out, replace the BatchNorm momentum with the default torch value, and freeze nothing — SimSiam updates every backbone weight. The detector head and neck are discarded for now and restored in `12_ssp_finetune.ipynb`.


In [ ]:
import torch, torch.nn as nn
from ultralytics import YOLO

SOURCE_WTS = os.path.join(DRIVE, "weights", "baseline_source", "best_source_v8s.pt")
assert os.path.isfile(SOURCE_WTS), f"Source weights missing: {SOURCE_WTS}. Run 05_train_source.ipynb first."

src = YOLO(SOURCE_WTS)
det_model = src.model

# YOLOv8 backbone is the first 10 modules in the YAML; SPPF is module 9 by default.
N_BACKBONE = 10
backbone_modules = list(det_model.model[:N_BACKBONE])
backbone = nn.Sequential(*backbone_modules)
print(f"REPORT backbone | modules = {len(backbone_modules)} "
      f"| trainable_params = {sum(p.numel() for p in backbone.parameters() if p.requires_grad):,}")


class SimSiam(nn.Module):
    def __init__(self, backbone, feat_dim, proj_dim=2048, pred_dim=512):
        super().__init__()
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, proj_dim), nn.BatchNorm1d(proj_dim), nn.ReLU(inplace=True),
            nn.Linear(proj_dim, proj_dim), nn.BatchNorm1d(proj_dim), nn.ReLU(inplace=True),
            nn.Linear(proj_dim, proj_dim), nn.BatchNorm1d(proj_dim, affine=False),
        )
        self.predictor = nn.Sequential(
            nn.Linear(proj_dim, pred_dim), nn.BatchNorm1d(pred_dim), nn.ReLU(inplace=True),
            nn.Linear(pred_dim, proj_dim),
        )

    def forward(self, x):
        h = self.backbone(x)
        h = self.pool(h).flatten(1)
        z = self.projector(h)
        p = self.predictor(z)
        return p, z.detach()


# Probe the backbone output channel count with a dry-run.
device = "cuda" if torch.cuda.is_available() else "cpu"
backbone = backbone.to(device).eval()
with torch.no_grad():
    feat = backbone(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device))
FEAT_DIM = feat.shape[1]
print(f"REPORT backbone_dim | feat = {tuple(feat.shape)} | channels = {FEAT_DIM}")

model = SimSiam(backbone, feat_dim=FEAT_DIM).to(device)
print(f"REPORT simsiam | total_params = {sum(p.numel() for p in model.parameters()):,}")


## 11.3 — Train SimSiam for 10 epochs (with resume-on-disconnect)

SimSiam loss is the negative cosine similarity between the predictor output and the stop-gradient projection of the other view, symmetrised across views. No labels, no negatives.

Hyperparameters follow Chen & He (2021):
- Optimizer: SGD with momentum 0.9, weight decay 1e-4.
- Learning rate: 0.05 × batch / 256 = 0.0125 at batch 64, cosine schedule.
- 10 epochs (paper uses 100; we use 10 because we start from `best_source_v8s.pt` rather than random init, and the Colab Free idle window can drop a 20-epoch run mid-way).

**Resume.** After every epoch the model + optimizer + scheduler + epoch index are written to `<DRIVE>/weights/ssp/ssp_ckpt.pt`. If you reconnect after a disconnect, just re-run all cells — this cell picks up where it stopped and only the in-progress epoch is lost.

**Force a clean run.** Set `FORCE_FRESH = True` in the cell below before running. It deletes the checkpoint and loss-log on Drive so training starts from epoch 0, regardless of any prior state.


In [ ]:
import torch.nn.functional as F
import csv, time

EPOCHS = 10
BASE_LR = 0.05 * BATCH / 256.0
WD = 1e-4
SSP_DIR = os.path.join(DRIVE, "weights", "ssp")
os.makedirs(SSP_DIR, exist_ok=True)
CKPT_PATH = os.path.join(SSP_DIR, "ssp_ckpt.pt")
LOSS_CSV  = os.path.join(SSP_DIR, "ssp_loss_log.csv")

# Flip this to True to ignore any existing checkpoint and start fresh.
# Useful when you changed hyperparameters or just want a clean curve.
FORCE_FRESH = False


def simsiam_loss(p1, z2, p2, z1):
    return -(F.cosine_similarity(p1, z2, dim=-1).mean()
             + F.cosine_similarity(p2, z1, dim=-1).mean()) / 2.0


optimizer = torch.optim.SGD(model.parameters(), lr=BASE_LR, momentum=0.9, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

start_epoch = 0
if FORCE_FRESH:
    for p in (CKPT_PATH, LOSS_CSV):
        if os.path.isfile(p):
            os.remove(p); print(f"FORCE_FRESH: removed {p}")
elif os.path.isfile(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=device)
    if ck.get("epochs_target") == EPOCHS:
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        start_epoch = ck["epoch"]
        print(f"RESUME from epoch {start_epoch}/{EPOCHS} (checkpoint found)")
    else:
        print(f"checkpoint targets EPOCHS={ck.get('epochs_target')} != {EPOCHS}; ignoring (training fresh)")

if start_epoch == 0:
    with open(LOSS_CSV, "w", newline="") as fh:
        csv.writer(fh).writerow(["epoch", "avg_loss", "lr", "elapsed_s"])

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    model.train()
    losses = []
    for v1, v2 in loader:
        v1 = v1.to(device, non_blocking=True)
        v2 = v2.to(device, non_blocking=True)
        p1, z1 = model(v1)
        p2, z2 = model(v2)
        loss = simsiam_loss(p1, z2, p2, z1)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    avg = sum(losses) / max(1, len(losses))
    dt = time.time() - t0
    print(f"epoch {epoch+1:02d}/{EPOCHS} | loss = {avg:+.4f} | lr = {scheduler.get_last_lr()[0]:.5f} | {dt:.1f}s")
    with open(LOSS_CSV, "a", newline="") as fh:
        csv.writer(fh).writerow([epoch + 1, round(avg, 4), round(scheduler.get_last_lr()[0], 5), round(dt, 1)])
    torch.save({
        "epoch": epoch + 1,
        "epochs_target": EPOCHS,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
    }, CKPT_PATH)

print("REPORT step = SSP_TRAIN | status = done")


## 11.4 — Save adapted backbone weights

The output is a state dict that `12_ssp_finetune.ipynb` will copy into a fresh YOLOv8s. We save only the backbone state dict (the projection head and predictor are SSP-only and not reused).


In [ ]:
BACKBONE_OUT = os.path.join(SSP_DIR, "backbone_simsiam.pt")
torch.save(model.backbone.state_dict(), BACKBONE_OUT)
size_mb = os.path.getsize(BACKBONE_OUT) / 1024 / 1024
print(f"REPORT backbone_out | path = {BACKBONE_OUT} | size_mb = {size_mb:0.1f}")

# Mirror to repo for the publish step.
REPO_TBL = "/content/visclick/reports/tables"
REPO_WTS = "/content/visclick/weights/ssp"
os.makedirs(REPO_TBL, exist_ok=True)
os.makedirs(REPO_WTS, exist_ok=True)
import shutil
shutil.copy2(LOSS_CSV, os.path.join(REPO_TBL, "ssp_loss_log.csv"))
# Backbone weights are too large for git; we publish only the CSV log.


## 11.5 — Publish loss log to git

The backbone `.pt` itself stays on Drive (too large for the repo). The training loss CSV is what we commit so the report's Figure showing the SSP loss curve has a regeneratable source on disk.

**Before running the cell below**, paste your GitHub personal-access token (PAT) into the `TOKEN = "PASTE_GITHUB_TOKEN_HERE"` line. The token lives only in Colab runtime memory and disappears when the runtime is recycled.


In [ ]:
# ---------------------------------------------------------------------------
# Paste your GitHub personal-access token (PAT) on the line below before
# running this cell. Replace the placeholder string. The token is NOT
# committed to git or saved anywhere; it lives only in the Colab runtime
# memory for this session, and disappears when the runtime is recycled.
# ---------------------------------------------------------------------------
TOKEN = "PASTE_GITHUB_TOKEN_HERE"

import os, subprocess

REPO_ROOT = "/content/visclick"
ARTIFACTS = [
    'reports/tables/ssp_loss_log.csv',
]

assert TOKEN and TOKEN != "PASTE_GITHUB_TOKEN_HERE", (
    "Paste your GitHub personal-access token into the TOKEN variable above "
    "before running this cell."
)

for rel in ARTIFACTS:
    p = os.path.join(REPO_ROOT, rel)
    assert os.path.exists(p), f"Missing artifact in repo clone: {p}. Run the previous section first."
    print(f"OK  {p}  ({os.path.getsize(p)} bytes)")


def run(cmd, **kw):
    r = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print("STDOUT:", r.stdout)
        print("STDERR:", r.stderr)
        raise RuntimeError(f"git command failed: {' '.join(cmd)}")
    return r.stdout


run(["git", "config", "user.email", "hiran@iit.ac.lk"])
run(["git", "config", "user.name",  "Hiran Abeywardhana"])

run(["git", "add", *ARTIFACTS])

status = run(["git", "status", "--porcelain"])
if not status.strip():
    print("REPORT step = GIT_PUBLISH | status = NOTHING_TO_COMMIT")
else:
    run(["git", "commit", "-m", 'D-02: SimSiam SSP pretrain loss log'])
    url = f"https://{TOKEN}@github.com/HiranMadhu/visclick.git"
    push = subprocess.run(["git", "push", url, "HEAD:main"],
                          cwd=REPO_ROOT, capture_output=True, text=True)
    if push.returncode != 0:
        print("PUSH STDERR:", push.stderr)
        raise RuntimeError("git push failed")
    print("REPORT step = GIT_PUBLISH | status = PUSHED")
